In [ ]:
MODEL_NAME = "bgfmamba"
SEEDS = [42, 123, 456, 789, 2024]
REP_SEED = 42
EPOCHS = 100
BATCH_SIZE = 4
IMAGE_SIZE = 128
LR = 0.001
LOSS_NAME = "lovasz_focal"
NUM_CLASSES = 1

In [ ]:
import os
if not os.getcwd().endswith("flood_mamba_edge"):
    if not os.path.exists("flood_mamba_edge"):
        !git clone https://github.com/doantrongthai/flood_mamba_edge.git
    %cd flood_mamba_edge
!pip install -q -r requirements.txt
!pip install -q tabulate gdown pandas

In [ ]:
import os

AUTO_PRETRAINED_MODELS = {'deeplabv3'}
IS_PRETRAINED = MODEL_NAME.lower() in AUTO_PRETRAINED_MODELS
PRETRAINED_FLAG = "--pretrained" if IS_PRETRAINED else ""
SEEDS_STR = " ".join(map(str, SEEDS))

def get_checkpoint_path(model, loss, dataset, seed):
    clean_loss = loss.lower().replace('-', '_')
    candidates = [
        f"outputs/seed_{seed}/{model}_{clean_loss}_{dataset}_s{seed}.pth",
        f"outputs/{model}_{clean_loss}_{dataset}_s{seed}.pth",
        f"outputs/seed_{seed}/{model}_{loss}_{dataset}_s{seed}.pth",
        f"outputs/{model}_{loss}_{dataset}_s{seed}.pth"
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return candidates[0]

In [ ]:
!python main.py --dataset floodkaggle --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --multiseed --seeds {SEEDS_STR} {PRETRAINED_FLAG} --download

In [ ]:
!python main.py --dataset aifloodsense --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --multiseed --seeds {SEEDS_STR} {PRETRAINED_FLAG} --download

In [ ]:
weights_fk = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "floodkaggle", REP_SEED)
!python cross_dataset_eval.py --model {MODEL_NAME} --weights {weights_fk} --source_dataset floodkaggle --target_datasets floodkaggle aifloodsense floodvn --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --download

In [ ]:
weights_afs = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "aifloodsense", REP_SEED)
!python cross_dataset_eval.py --model {MODEL_NAME} --weights {weights_afs} --source_dataset aifloodsense --target_datasets aifloodsense floodkaggle floodvn --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --download

In [ ]:
import shutil
try:
    from google.colab import files
except Exception:
    files = None

bundle_dir = f"{MODEL_NAME}_All_Artifacts"
os.makedirs(bundle_dir, exist_ok=True)
output_dir = "outputs"

if os.path.exists(output_dir):
    for item in sorted(os.listdir(output_dir)):
        src = os.path.join(output_dir, item)
        dst = os.path.join(bundle_dir, item)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        elif os.path.isfile(src):
            shutil.copy2(src, dst)

zip_filename = f"{MODEL_NAME}_artifacts.zip"
if os.path.exists(zip_filename):
    os.remove(zip_filename)

shutil.make_archive(f"{MODEL_NAME}_artifacts", "zip", bundle_dir)
if files is not None:
    try:
        files.download(zip_filename)
    except Exception:
        pass